In [8]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor


In [2]:
df = pd.read_csv(r"C:\Users\Aanjney\Desktop\Deploy\VGSALES\vgsales.csv")
df.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [3]:
df.shape

(16598, 11)

In [4]:
df.columns

Index(['Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales',
       'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales'],
      dtype='object')

In [5]:
df.describe()

,Rank,Year,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
count,16598.000000,16327.000000,16598.000000,16598.000000,16598.000000,16598.000000,16598.000000
mean,8300.605254,2006.406443,0.264667,0.146652,0.077782,0.048063,0.537441
std,4791.853933,5.828981,0.816683,0.505351,0.309291,0.188588,1.555028
min,1.000000,1980.000000,0.000000,0.000000,0.000000,0.000000,0.010000
25%,4151.250000,2003.000000,0.000000,0.000000,0.000000,0.000000,0.060000
50%,8300.500000,2007.000000,0.080000,0.020000,0.000000,0.010000,0.170000
75%,12449.750000,2010.000000,0.240000,0.110000,0.040000,0.040000,0.470000
max,16600.000000,2020.000000,41.490000,29.020000,10.220000,10.570000,82.740000


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  object 
 2   Platform      16598 non-null  object 
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  object 
 5   Publisher     16540 non-null  object 
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), object(4)
memory usage: 1.4+ MB


In [7]:
df.isnull().sum()

Rank              0
Name              0
Platform          0
Year            271
Genre             0
Publisher        58
NA_Sales          0
EU_Sales          0
JP_Sales          0
Other_Sales       0
Global_Sales      0
dtype: int64

In [9]:
df.dropna(inplace=True)

In [10]:
features = ['Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales', 'EU_Sales', 'JP_Sales']
target = 'Global_Sales'

X = df[features]
y = df[target]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
categorical_cols = ['Platform', 'Genre', 'Publisher']
numerical_cols = ['Year', 'NA_Sales', 'EU_Sales', 'JP_Sales']

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

In [14]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "DecisionTree": DecisionTreeRegressor(),
    "RandomForest": RandomForestRegressor()
}

param_grid = {
    "Ridge": {"model__alpha": [0.1, 1.0, 10]},
    "Lasso": {"model__alpha": [0.01, 0.1, 1]},
    "DecisionTree": {"model__max_depth": [None, 10, 20]},
    "RandomForest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 10]
    }
}

best_score = -np.inf
best_model = None

In [18]:
for name, model in models.items():
    
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    if name in param_grid:
        grid = GridSearchCV(
            pipeline,
            param_grid[name],
            cv=5,
            scoring='r2',
            n_jobs=-1
        )
        grid.fit(X_train, y_train)
        model_pipeline = grid.best_estimator_
    else:
        model_pipeline = pipeline.fit(X_train, y_train)
    
    cv_score = cross_val_score(model_pipeline, X_train, y_train, cv=5, scoring='r2').mean()
    
    y_pred = model_pipeline.predict(X_test)
    test_score = r2_score(y_test, y_pred)
    
    print(f"{name}")
    print("CV Score:", cv_score)
    print("Test R2:", test_score)
    print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
    print("----------")
    
    if test_score > best_score:
        best_score = test_score
        best_model = model_pipeline

LinearRegression
CV Score: 0.991119071973445
Test R2: 0.9952976741451816
RMSE: 0.14183228954865837
----------
Ridge
CV Score: 0.9911346358190796
Test R2: 0.9953028544896012
RMSE: 0.14175414283981835
----------
Lasso
CV Score: 0.990829797511284
Test R2: 0.9950762052287663
RMSE: 0.1451338528419432
----------
DecisionTree
CV Score: 0.9274409084116868
Test R2: 0.8535325720374481
RMSE: 0.7915699168756392
----------
RandomForest
CV Score: 0.9582964086391842
Test R2: 0.8150561910573639
RMSE: 0.8894850478147519
----------


In [19]:
pickle.dump(best_model, open("best_model.pkl", "wb"))

print("Best model saved successfully!")

Best model saved successfully!
